# MobileNet — standard/robust comparison

Дообучает отдельного кандидата от текущего MobileNet-чемпиона. Standard registry не изменяется; полный robust-кандидат может обновить только `registry/robust`.

In [ ]:
REPO_URL = "https://github.com/frest1ler/text-orientation-classification.git"
BRANCH = "main"
PROJECT_DIR = "/content/drive/MyDrive/text-orientation"
AUGMENTATION_PROFILE = "robust"  # standard | robust
QUICK_RUN = True
EPOCHS = 3
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 64
VALIDATION_BATCH_SIZE = 128
NUM_WORKERS = 2
RESUME_TRAINING = True
RUN_TESTS = True
PROMOTE_ROBUST_CHAMPION = not QUICK_RUN and AUGMENTATION_PROFILE == "robust"

In [ ]:
import os, subprocess, sys
from pathlib import Path

repo_dir = Path("/content/text-orientation-classification")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json, torch
if AUGMENTATION_PROFILE not in {"standard", "robust"}:
    raise ValueError("AUGMENTATION_PROFILE must be 'standard' or 'robust'")
if not torch.cuda.is_available():
    raise RuntimeError("В Colab выберите Runtime → Change runtime type → GPU")
print("gpu:", torch.cuda.get_device_name(0))
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
from src.registry import select_champion
registry = Path(PROJECT_DIR) / "registry"
bundle = select_champion(registry, "mobilenet_v3_large")
initial_checkpoint = bundle.checkpoint_path
print("initial checkpoint:", initial_checkpoint)

In [ ]:
from IPython.display import Image as DisplayImage, display
preview = Path("/content/robust_preview.png")
subprocess.run([
    sys.executable, "-m", "scripts.preview_synthetic",
    "--augmentation-profile", AUGMENTATION_PROFILE,
    "--pairs", "8", "--epoch", "1", "--output", str(preview),
], check=True)
display(DisplayImage(filename=str(preview)))

In [ ]:
mode = "quick" if QUICK_RUN else "full"
run_name = f"mobilenet_{AUGMENTATION_PROFILE}_{mode}"
run_dir = Path("artifacts/experiments") / run_name
recovery_dir = Path(PROJECT_DIR) / "training/recovery" / AUGMENTATION_PROFILE / "mobilenet_v3_large" / mode
command = [
    sys.executable, "-m", "scripts.train_robust",
    "--output-dir", str(run_dir),
    "--recovery-dir", str(recovery_dir),
    "--augmentation-profile", AUGMENTATION_PROFILE,
    "--initial-checkpoint", str(initial_checkpoint),
    "--epochs", str(2 if QUICK_RUN else EPOCHS),
    "--learning-rate", str(LEARNING_RATE),
    "--batch-size", str(TRAIN_BATCH_SIZE),
    "--validation-batch-size", str(VALIDATION_BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
]
if QUICK_RUN:
    command += ["--train-base-samples", "2048", "--validation-base-samples", "512"]
if RESUME_TRAINING:
    command.append("--resume")
subprocess.run(command, check=True)

In [ ]:
import platform, shutil
from datetime import datetime, timezone

environment = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
}
(run_dir / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
if not QUICK_RUN:
    subprocess.run([sys.executable, "-m", "scripts.calibrate", "--run-dir", str(run_dir), "--config", "configs/baseline.yaml"], check=True)
    if PROMOTE_ROBUST_CHAMPION:
        subprocess.run([sys.executable, "-m", "scripts.promote_robust_champion", "--run-dir", str(run_dir), "--project-dir", PROJECT_DIR], check=True)
        subprocess.run([sys.executable, "-m", "scripts.compare_robust", "--project-dir", PROJECT_DIR, "--minimum-improvement", "0.005"], check=True)
runs_dir = Path(PROJECT_DIR) / "training/runs" / AUGMENTATION_PROFILE / "mobilenet_v3_large" / mode
runs_dir.mkdir(parents=True, exist_ok=True)
archive = Path(shutil.make_archive(f"/content/{run_name}", "zip", root_dir=run_dir))
destination = runs_dir / archive.name
shutil.copy2(archive, destination)
print("Готово:", destination)
print("Standard registry не изменён.")